# Digit Classification using TensorFlow 2 — Pure TensorFlow Model

This notebook demonstrates MNIST digit classification using **TensorFlow 2**.

### Important design
- **Keras is used only to load the MNIST dataset:** `tf.keras.datasets.mnist.load_data()`
- The model is built manually using **core TensorFlow**
- No `Sequential`
- No `Dense`
- No `model.fit()`
- No `model.compile()`
- No `model.evaluate()`

The purpose is to understand what happens inside a simple neural-network training workflow.

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

## 1. Load the MNIST dataset

We use the convenient TensorFlow Keras dataset loader **only for obtaining the MNIST data**.

MNIST contains:
- 60,000 training images
- 10,000 test images
- Each image is 28 × 28 pixels
- There are 10 classes: digits 0 through 9

The model itself will not use any high-level model-building API.

In [ ]:
# Keras is used ONLY for loading the dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)
print("Test images:", x_test.shape)
print("Test labels:", y_test.shape)

## 2. Inspect one image

Each MNIST image is initially a 28 × 28 grayscale image.

The label tells us which digit is present.

In [ ]:
plt.figure(figsize=(4, 4))
plt.imshow(x_train[0], cmap="gray")
plt.title("Label: {}".format(y_train[0]))
plt.axis("off")
plt.show()

## 3. Check the pixel values

MNIST images contain grayscale pixel values from 0 to 255.

Before training, we normalize these values to the range 0 to 1.

In [ ]:
print("Minimum pixel value:", x_train.min())
print("Maximum pixel value:", x_train.max())

## 4. Normalize the images

We divide every pixel by 255.

This changes the range:

`0 → 0.0`

`255 → 1.0`

Normalization generally makes numerical optimization easier.

In [ ]:
x_train = x_train.astype(np.float32) / 255.0
x_test = x_test.astype(np.float32) / 255.0

print("New minimum:", x_train.min())
print("New maximum:", x_train.max())

## 5. Flatten the images

Our model will use a simple linear classifier.

A 28 × 28 image contains:

`28 × 28 = 784`

pixel values.

Therefore, each image is converted from:

`(28, 28) → (784,)`

This allows us to perform matrix multiplication with the weight matrix.

In [ ]:
x_train = x_train.reshape(-1, 784)
x_test = x_test.reshape(-1, 784)

print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)

## 6. Convert the data to TensorFlow tensors

The dataset was loaded into NumPy arrays.

We now convert the feature and label arrays into TensorFlow tensors so TensorFlow operations can work directly with them.

In [ ]:
x_train = tf.convert_to_tensor(x_train, dtype=tf.float32)
y_train = tf.convert_to_tensor(y_train, dtype=tf.int32)

x_test = tf.convert_to_tensor(x_test, dtype=tf.float32)
y_test = tf.convert_to_tensor(y_test, dtype=tf.int32)

print(x_train.dtype)
print(y_train.dtype)

## 7. Define the training configuration

We will train using mini-batches.

- `batch_size`: number of images processed in one step
- `n_steps`: number of parameter-update steps
- `learning_rate`: controls how large each update is

In [ ]:
batch_size = 30
n_steps = 10000
learning_rate = 0.01

print("Batch size:", batch_size)
print("Training steps:", n_steps)
print("Learning rate:", learning_rate)

## 8. Create the model parameters

We are building the model manually.

There are:
- 784 input features
- 10 output classes

Therefore:

`W` has shape `(784, 10)`

`b` has shape `(10,)`

These are TensorFlow variables because their values must change during training.

In [ ]:
W = tf.Variable(
    tf.zeros([784, 10]),
    name="weights"
)

b = tf.Variable(
    tf.zeros([10]),
    name="bias"
)

print("W shape:", W.shape)
print("b shape:", b.shape)

## 9. Understand the model mathematically

The forward pass is:

`logits = XW + b`

For one image:
- `X` has 784 values
- `W` maps 784 inputs to 10 outputs
- `b` provides one bias value for each class

The result contains 10 scores, one for each digit class.

These scores are called **logits**.

In [ ]:
def model(X):
    return tf.matmul(X, W) + b

## 10. Test the forward pass

Before training, we can pass a small batch through the model.

Because the model has 10 output classes, the output for a batch of 5 images should have shape `(5, 10)`.

In [ ]:
sample_logits = model(x_train[:5])

print("Input shape:", x_train[:5].shape)
print("Output shape:", sample_logits.shape)
print(sample_logits)

## 11. Define the loss function

We use sparse softmax cross-entropy.

`tf.nn.sparse_softmax_cross_entropy_with_logits` expects:
- integer class labels such as `0, 1, 2, ..., 9`
- raw logits from the model

The function internally handles the softmax-related calculation and computes the classification loss.

In [ ]:
def compute_loss(X, y):
    logits = model(X)

    per_example_loss = tf.nn.sparse_softmax_cross_entropy_with_logits(
        labels=y,
        logits=logits
    )

    return tf.reduce_mean(per_example_loss)

## 12. Check the initial loss

The weights and bias currently contain zeros, so the model has not learned anything yet.

We can calculate the initial loss before training.

In [ ]:
initial_loss = compute_loss(x_train[:batch_size], y_train[:batch_size])

print("Initial loss:", float(initial_loss.numpy()))

## 13. Create the optimizer

We use TensorFlow's Adam optimizer.

The optimizer receives the gradients and updates `W` and `b` to reduce the loss.

This uses the TensorFlow optimizer API directly; there is no high-level model training API involved.

In [ ]:
optimizer = tf.optimizers.Adam(
    learning_rate=learning_rate
)

print(optimizer)

## 14. Create the TensorFlow training step

`tf.GradientTape()` records the operations used to calculate the loss.

Then:

1. Calculate the loss
2. Calculate gradients of the loss with respect to `W` and `b`
3. Apply the gradients using Adam

This is the core of manual TensorFlow training.

In [ ]:
def train_step(X_batch, y_batch):
    with tf.GradientTape() as tape:
        loss_value = compute_loss(X_batch, y_batch)

    gradients = tape.gradient(
        loss_value,
        [W, b]
    )

    optimizer.apply_gradients(
        zip(gradients, [W, b])
    )

    return loss_value

## 15. Create a TensorFlow data pipeline

`tf.data.Dataset` creates batches from the training data.

The pipeline:
- stores `(image, label)` pairs
- shuffles the training examples
- creates mini-batches
- repeats the dataset continuously

This lets us request the next training batch during every training step.

In [ ]:
train_dataset = tf.data.Dataset.from_tensor_slices(
    (x_train, y_train)
)

train_dataset = train_dataset.shuffle(
    buffer_size=x_train.shape[0],
    reshuffle_each_iteration=True
)

train_dataset = train_dataset.batch(batch_size).repeat()

train_iterator = iter(train_dataset)

print("TensorFlow training dataset is ready.")

## 16. Inspect one training batch

Before starting the full training loop, let's inspect the shape of one batch.

With `batch_size = 30`:
- Images: `(30, 784)`
- Labels: `(30,)`

In [ ]:
batch_X, batch_y = next(train_iterator)

print("Batch images:", batch_X.shape)
print("Batch labels:", batch_y.shape)

## 17. Run one training step

This is a single parameter update.

The loss is calculated, gradients are computed using `GradientTape`, and Adam updates the weights and bias.

In [ ]:
loss_value = train_step(batch_X, batch_y)

print("Loss after one training step:", float(loss_value.numpy()))

## 18. Train the model

Now we repeat the training step.

Every iteration:
1. Get the next mini-batch
2. Calculate the loss
3. Compute gradients
4. Update `W` and `b`

The loss is printed periodically so we can observe the training process.

In [ ]:
loss_history = []

for step in range(n_steps):
    batch_X, batch_y = next(train_iterator)

    loss_value = train_step(batch_X, batch_y)
    loss_history.append(float(loss_value.numpy()))

    if (step + 1) % 1000 == 0:
        print(
            "Step = {}, Loss = {:.4f}".format(
                step + 1,
                float(loss_value.numpy())
            )
        )

## 19. Plot the training loss

The loss should generally decrease as the model learns from the training data.

A decreasing loss indicates that the optimization process is finding parameter values that better fit the training examples.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(loss_history)
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()

## 20. Generate predictions

The trained model produces 10 logits for each test image.

The predicted digit is the class with the largest logit.

We use `tf.argmax` to find that class.

In [ ]:
test_logits = model(x_test)

predicted_labels = tf.argmax(
    test_logits,
    axis=1,
    output_type=tf.int32
)

print("Predicted labels:", predicted_labels[:20].numpy())

## 21. Calculate accuracy

We compare:
- predicted labels
- actual labels

`tf.equal` gives `True` when the prediction is correct.

We convert those Boolean values to numbers and calculate their mean.

In [ ]:
correct_predictions = tf.equal(
    predicted_labels,
    y_test
)

accuracy = tf.reduce_mean(
    tf.cast(correct_predictions, tf.float32)
)

accuracy_value = accuracy.numpy()

print("Test Accuracy = {:.3f}".format(accuracy_value))

## 22. Display sample predictions

Let's visually inspect a few test images and compare the predicted digit with the actual label.

In [ ]:
plt.figure(figsize=(12, 6))

for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(tf.reshape(x_test[i], [28, 28]), cmap="gray")
    plt.title(
        "Actual: {} | Pred: {}".format(
            int(y_test[i].numpy()),
            int(predicted_labels[i].numpy())
        )
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

## 23. Inspect the learned parameters

After training, `W` and `b` contain learned values.

`W` represents how strongly each input pixel contributes to each output class.

`b` represents the learned bias for each digit class.

In [ ]:
print("Weights shape:", W.shape)
print("Bias shape:", b.shape)

print("First 5 bias values:")
print(b[:5].numpy())

# TensorFlow 1.x → TensorFlow 2 mapping

The original TensorFlow 1.x-style exercise used concepts such as:

| TensorFlow 1.x | TensorFlow 2 approach |
|---|---|
| `tf.Session()` | Eager execution |
| `tf.placeholder()` | Function arguments / tensors |
| `feed_dict` | Direct tensor input |
| `tf.global_variables_initializer()` | Variables initialize directly |
| `tf.train.AdamOptimizer` | `tf.optimizers.Adam` |
| Graph execution | Immediate/eager execution |
| Manual session `run()` | Direct TensorFlow operations |

The important idea is that TensorFlow 2 lets us write the mathematical operations directly while still giving us access to low-level control over variables, gradients, and optimization.

# Complete learning flow

The complete workflow in this notebook is:

**MNIST dataset**

↓  

**Normalize pixels**

↓  

**Flatten 28 × 28 → 784**

↓  

**TensorFlow Variables: W and b**

↓  

**Forward pass: `XW + b`**

↓  

**Logits**

↓  

**Sparse Softmax Cross-Entropy**

↓  

**GradientTape**

↓  

**Gradients**

↓  

**Adam optimizer**

↓  

**Updated W and b**

↓  

**Predictions**

↓  

**Accuracy**

This is a manually implemented TensorFlow training workflow without using a high-level model-building API.

# Connection with the Perceptron

This model is closely related to the perceptron concept.

For one output:

`z = w₁x₁ + w₂x₂ + ... + b`

For MNIST, we have 784 input pixels and 10 output classes:

`Z = XW + b`

The important difference is that instead of producing only a single binary output, this model produces **10 class scores**.

This provides a useful bridge from:

**Perceptron → Linear Classifier → Neural Networks → Deep Learning**.

# Key takeaway

The main purpose of this notebook is not just to classify MNIST.

It is to understand the fundamental TensorFlow components behind model training:

- `tf.Variable`
- `tf.matmul`
- TensorFlow loss functions
- `tf.GradientTape`
- TensorFlow optimizers
- `tf.data`
- `tf.argmax`
- `tf.reduce_mean`

Only the **MNIST dataset loader** comes from `tf.keras.datasets`.

The actual model and training logic are implemented directly with TensorFlow.